In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score

In [6]:
prop_df_celltypist = pd.read_csv('/ibex/user/sotoorda/masterh1/LLM_project/CellTypist/predictions_celltypist.csv')
prop_df_rf = pd.read_csv('/ibex/user/sotoorda/masterh1/LLM_project/Random_forest/prediction_results.csv')
prop_df_seurat = pd.read_csv('/ibex/user/sotoorda/masterh1/LLM_project/Label_transfer/predictions_seurat.csv')
prop_df_scgpt = pd.read_csv('/ibex/user/sotoorda/masterh1/LLM_project/scGPT/scgpt_prob_with_labels.csv')

In [10]:
from collections import Counter
from pathlib import Path
import json

bench_dir = Path('/ibex/user/sotoorda/masterh1/LLM_project/benchmarking')
bench_dir.mkdir(parents=True, exist_ok=True)


def _model_view(df, prediction_col, confidence_col, model_name):
    required = ['Cell_id', 'Ground_truth', prediction_col]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f'{model_name} is missing required columns: {missing}')

    view = df[required].copy()
    view = view.rename(columns={prediction_col: model_name})

    if confidence_col in df.columns:
        confidence = pd.to_numeric(df[confidence_col], errors='coerce')
    else:
        excluded = {'Cell_id', 'Ground_truth', prediction_col, confidence_col}
        numeric_cols = [col for col in df.columns if col not in excluded and pd.api.types.is_numeric_dtype(df[col])]
        if not numeric_cols:
            raise KeyError(f'{model_name} has no numeric probability columns to derive confidence from.')
        confidence = df[numeric_cols].max(axis=1)

    view[f'{model_name}_confidence'] = confidence
    return view


def _merge_on_cell_id(frames):
    merged = frames[0]
    for frame in frames[1:]:
        merged = merged.merge(frame, on='Cell_id', how='inner', suffixes=('', '_dup'))
    return merged


def _agreement_class(vote_support):
    return {1: 'disagreement', 2: 'majority', 3: 'unanimous'}[int(vote_support)]


def _ensemble_from_row(row):
    predictions = {
        'CellTypist': row['CellTypist'],
        'RandomForest': row['RandomForest'],
        'Seurat': row['Seurat'],
    }
    counts = Counter(predictions.values())
    vote_support = max(counts.values())

    if vote_support == 1:
        confidence_lookup = {
            label: row[f'{model}_confidence']
            for model, label in predictions.items()
        }
        ensemble_prediction = max(confidence_lookup.items(), key=lambda item: item[1])[0]
    else:
        ensemble_prediction = counts.most_common(1)[0][0]

    return pd.Series({
        'Ensemble_prediction': ensemble_prediction,
        'Ensemble_vote_support': vote_support,
        'Ensemble_confidence': vote_support / 3.0,
        'Agreement_class': _agreement_class(vote_support),
    })


def _metrics(y_true, y_pred):
    labels = sorted(set(y_true.astype(str)) | set(y_pred.astype(str)))
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    }


celltypist_view = _model_view(prop_df_celltypist, 'Predicted_label', 'Prediction_prob', 'CellTypist')
rf_view = _model_view(prop_df_rf, 'Predicted_label', 'Prediction_prob', 'RandomForest')
seurat_view = _model_view(prop_df_seurat, 'Predicted_label', 'prediction.score.max', 'Seurat')
scgpt_view = _model_view(prop_df_scgpt, 'scGPT_prediction', 'scGPT_confidence', 'scGPT')
scgpt_view = scgpt_view.rename(columns={'scGPT': 'scGPT_prediction'})

aligned = celltypist_view.merge(rf_view, on=['Cell_id', 'Ground_truth'], how='inner')
aligned = aligned.merge(seurat_view, on=['Cell_id', 'Ground_truth'], how='inner')
aligned = aligned.merge(scgpt_view, on=['Cell_id', 'Ground_truth'], how='inner')

aligned = aligned[[
    'Cell_id',
    'Ground_truth',
    'CellTypist',
    'CellTypist_confidence',
    'RandomForest',
    'RandomForest_confidence',
    'Seurat',
    'Seurat_confidence',
    'scGPT_prediction',
    'scGPT_confidence',
]]

ensemble_components = aligned[['Cell_id', 'Ground_truth', 'CellTypist', 'RandomForest', 'Seurat', 'CellTypist_confidence', 'RandomForest_confidence', 'Seurat_confidence']].copy()
ensemble_details = ensemble_components.apply(_ensemble_from_row, axis=1)
ensemble_ready = pd.concat([
    ensemble_components[['Cell_id', 'Ground_truth', 'CellTypist', 'RandomForest', 'Seurat']],
    ensemble_details,
], axis=1)

ensemble_ready = ensemble_ready[[
    'Cell_id',
    'Ground_truth',
    'Ensemble_prediction',
    'Ensemble_vote_support',
    'Ensemble_confidence',
    'CellTypist',
    'RandomForest',
    'Seurat',
    'Agreement_class',
]]

scgpt_ready = aligned[['Cell_id', 'Ground_truth', 'scGPT_prediction', 'scGPT_confidence']].copy()

merged = ensemble_ready.merge(
    scgpt_ready,
    on=['Cell_id', 'Ground_truth'],
    how='inner',
)
merged['models_agree'] = merged['Ensemble_prediction'].astype(str) == merged['scGPT_prediction'].astype(str)
merged['ensemble_correct'] = merged['Ensemble_prediction'].astype(str) == merged['Ground_truth'].astype(str)
merged['scgpt_correct'] = merged['scGPT_prediction'].astype(str) == merged['Ground_truth'].astype(str)

benchmark_rows = []
for model_name, prediction_col, confidence_col in [
    ('CellTypist', 'CellTypist', 'CellTypist_confidence'),
    ('RandomForest', 'RandomForest', 'RandomForest_confidence'),
    ('Seurat', 'Seurat', 'Seurat_confidence'),
    ('Ensemble', 'Ensemble_prediction', 'Ensemble_confidence'),
    ('scGPT', 'scGPT_prediction', 'scGPT_confidence'),
]:
    metrics = _metrics(merged['Ground_truth'].astype(str), merged[prediction_col].astype(str))
    row = {
        'model': model_name,
        **metrics,
        'n_cells': int(len(merged)),
    }
    if model_name != 'scGPT':
        row['agreement_with_scGPT_pct'] = float((merged[prediction_col].astype(str) == merged['scGPT_prediction'].astype(str)).mean() * 100.0)
    else:
        row['agreement_with_scGPT_pct'] = 100.0
    benchmark_rows.append(row)

benchmark_df = pd.DataFrame(benchmark_rows)

ensemble_ready.to_csv(bench_dir / 'ensemble_benchmark_ready.csv', index=False)
scgpt_ready.to_csv(bench_dir / 'scgpt_benchmark_ready.csv', index=False)
aligned.to_csv(bench_dir / 'aligned_model_predictions.csv', index=False)
merged.to_csv(bench_dir / 'ensemble_vs_scgpt_merged.csv', index=False)
benchmark_df.to_csv(bench_dir / 'model_vs_scgpt_metrics.csv', index=False)

summary = {
    'n_cells': int(len(merged)),
    'files': {
        'ensemble_benchmark_ready': str(bench_dir / 'ensemble_benchmark_ready.csv'),
        'scgpt_benchmark_ready': str(bench_dir / 'scgpt_benchmark_ready.csv'),
        'aligned_model_predictions': str(bench_dir / 'aligned_model_predictions.csv'),
        'ensemble_vs_scgpt_merged': str(bench_dir / 'ensemble_vs_scgpt_merged.csv'),
        'model_vs_scgpt_metrics': str(bench_dir / 'model_vs_scgpt_metrics.csv'),
    },
    'metrics': benchmark_df.to_dict(orient='records'),
}

with open(bench_dir / 'ensemble_vs_scgpt_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

print('Benchmark files written to:', bench_dir)
print(benchmark_df.sort_values('model').to_string(index=False))

/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/ibex/user/sotoorda/conda-environments/test_env/lib/python3.9/site-packages/sklearn/metrics/_classif

Benchmark files written to: /ibex/user/sotoorda/masterh1/LLM_project/benchmarking
       model  accuracy  balanced_accuracy  macro_precision  macro_recall  macro_f1  n_cells  agreement_with_scGPT_pct
  CellTypist  0.199437           0.291584         0.191171      0.112148  0.126170    23466                 63.129634
    Ensemble  0.208983           0.362670         0.228169      0.139488  0.161489    23466                 86.162959
RandomForest  0.183500           0.250519         0.273489      0.100207  0.114384    23466                 78.419841
      Seurat  0.226626           0.403133         0.330324      0.212175  0.233300    23466                 88.651666
       scGPT  0.206469           0.335288         0.280199      0.145777  0.156676    23466                100.000000
